In [8]:
import requests
r = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU", timeout=10)
print(r.status_code)
print(r.text[:200])

200
observation_date,DCOILBRENTEU
1987-05-20,18.63
1987-05-21,18.45
1987-05-22,18.55
1987-05-25,18.60
1987-05-26,18.63
1987-05-27,18.60
1987-05-28,18.60
1987-05-29,18.58
1987-06-01,18.65
1987-06-02,18.68



# External Data — Programmatic Download

This notebook downloads all external data required for the SVAR (Blanchard-Quah identification).

**Variables sourced:**
| Variable | Source | Frequency |
|---|---|---|
| Industrial production Ukraine | World Bank API | Monthly (interpolated from quarterly) |
| Industrial production Euro Area | FRED API | Monthly |
| Exchange rate UAH/USD | FRED API | Monthly |
| NBU policy rate | FRED API | Monthly |
| ECB refi rate | FRED API | Monthly |
| Brent crude oil price | FRED API | Monthly |

All series are saved to `data/data_external/` for use in the SVAR notebook.

In [7]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import requests
import pandas_datareader.data as web
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

cwd = Path.cwd()
DATA_EXTERNAL = cwd.parent / "data" / "data_external"
DATA_EXTERNAL.mkdir(parents=True, exist_ok=True)

START = "2005-01-01"
END   = "2025-12-31"

print("Output folder:", DATA_EXTERNAL)
print("Sample period:", START, "to", END)

ModuleNotFoundError: No module named 'distutils'

## 1. FRED API — Exchange rate, policy rates, oil price, EA industrial production

FRED (Federal Reserve Bank of St. Louis) provides free access to hundreds of macro series.
No API key required for basic access via `pandas_datareader`.

Series used:
- `DEXUSEU` — USD/EUR exchange rate (to derive UAH/EUR from UAH/USD)
- `DEXUAEU` — UAH/USD exchange rate
- `UKRAIRTT01STM` — NBU policy rate (Ukraine)
- `ECBDFR` — ECB deposit facility rate
- `DCOILBRENTEU` — Brent crude oil price (USD/barrel)
- `EA19PRINTO01GYSAM` — Euro Area industrial production YoY

In [ ]:
FRED_SERIES = {
    "DEXUAEU":            "uah_usd",        # UAH per USD
    "DEXUSEU":            "usd_eur",        # USD per EUR (to get UAH/EUR)
    "ECBDFR":             "ecb_rate",       # ECB deposit rate
    "DCOILBRENTEU":       "brent",          # Brent oil price
    "EA19PRINTO01GYSAM":  "ea_ip_yoy",      # EA industrial production YoY
}

fred_data = {}

for code, name in FRED_SERIES.items():
    try:
        s = web.DataReader(code, "fred", START, END)[code]
        # Resample to monthly if needed
        s = s.resample("MS").mean()
        s.index = s.index.to_period("M").to_timestamp()
        fred_data[name] = s
        print(f"✓ {name:20s} — {len(s.dropna())} obs | {s.dropna().index.min().date()} to {s.dropna().index.max().date()}")
    except Exception as e:
        print(f"✗ {name:20s} — FAILED: {e}")

print()
# Derive UAH/EUR
if "uah_usd" in fred_data and "usd_eur" in fred_data:
    fred_data["uah_eur"] = fred_data["uah_usd"] / fred_data["usd_eur"]
    print("✓ uah_eur derived from uah_usd / usd_eur")

## 2. NBU policy rate — National Bank of Ukraine API

The NBU provides a free JSON API. We fetch the official discount rate.

In [ ]:
try:
    url = "https://bank.gov.ua/NBU_statistic/interest_rates/diskont_rate.json"
    resp = requests.get(url, timeout=15)
    nbu_raw = pd.DataFrame(resp.json())
    nbu_raw["date"] = pd.to_datetime(nbu_raw["date_start"], dayfirst=True)
    nbu_raw = nbu_raw.sort_values("date").set_index("date")

    # Resample to monthly (take end-of-month value)
    date_range = pd.date_range(START, END, freq="MS")
    nbu_rate = nbu_raw["value"].reindex(date_range, method="ffill")
    nbu_rate.name = "nbu_rate"
    fred_data["nbu_rate"] = nbu_rate
    print(f"✓ nbu_rate            — {len(nbu_rate.dropna())} obs | {nbu_rate.dropna().index.min().date()} to {nbu_rate.dropna().index.max().date()}")
except Exception as e:
    print(f"✗ nbu_rate — NBU API failed: {e}")
    print("  Falling back to FRED series UKRAIRTT01STM")
    try:
        s = web.DataReader("UKRAIRTT01STM", "fred", START, END)["UKRAIRTT01STM"]
        s = s.resample("MS").mean()
        s.index = s.index.to_period("M").to_timestamp()
        fred_data["nbu_rate"] = s
        print(f"✓ nbu_rate (FRED)     — {len(s.dropna())} obs")
    except Exception as e2:
        print(f"✗ nbu_rate fallback also failed: {e2}")

## 3. Ukraine Industrial Production — World Bank API

Monthly industrial production data for Ukraine is not available on FRED.
We use the World Bank API (indicator `NV.IND.TOTL.KD.ZG` — industry value added growth)
as a proxy, then interpolate quarterly to monthly.

For robustness we also try the OECD API.

In [ ]:
ukr_ip = None

# --- Attempt 1: OECD API (monthly industrial production index) ---
try:
    url = (
        "https://stats.oecd.org/SDMX-JSON/data/MEI_REAL/UKR.PRINTO01.ST.M/all"
        "?startTime=2005-01&endTime=2025-12&contentType=csv"
    )
    resp = requests.get(url, timeout=20)
    if resp.status_code == 200 and len(resp.content) > 100:
        from io import StringIO
        df_oecd = pd.read_csv(StringIO(resp.text))
        # parse OECD CSV
        df_oecd["date"] = pd.to_datetime(df_oecd["TIME_PERIOD"], format="%Y-%m")
        ukr_ip = df_oecd.set_index("date")["OBS_VALUE"].rename("ukr_ip")
        print(f"✓ ukr_ip (OECD)       — {len(ukr_ip.dropna())} obs")
    else:
        raise ValueError(f"OECD returned status {resp.status_code}")
except Exception as e:
    print(f"  OECD attempt failed: {e}")

# --- Attempt 2: IMF IFS via SDMX ---
if ukr_ip is None:
    try:
        url = "https://www.imf.org/external/datamapper/api/v1/AIP_IX?periods=2005:2025&countries=UKR"
        resp = requests.get(url, timeout=20)
        data_imf = resp.json()
        values = data_imf["values"]["AIP_IX"]["UKR"]
        ukr_ip = pd.Series(values, name="ukr_ip")
        ukr_ip.index = pd.to_datetime(ukr_ip.index)
        ukr_ip = ukr_ip.resample("MS").mean()
        print(f"✓ ukr_ip (IMF)        — {len(ukr_ip.dropna())} obs")
    except Exception as e:
        print(f"  IMF attempt failed: {e}")

# --- Attempt 3: World Bank quarterly → interpolate to monthly ---
if ukr_ip is None:
    try:
        import wbdata
        wb_data = wbdata.get_dataframe(
            {"NV.IND.TOTL.KD.ZG": "ind_growth"},
            country="UKR"
        )
        wb_data.index = pd.to_datetime(wb_data.index)
        wb_data = wb_data.sort_index()
        # Interpolate annual to monthly
        monthly_idx = pd.date_range(wb_data.index.min(), wb_data.index.max(), freq="MS")
        ukr_ip = wb_data["ind_growth"].reindex(
            wb_data.index.union(monthly_idx)
        ).interpolate(method="time").reindex(monthly_idx)
        ukr_ip.name = "ukr_ip"
        print(f"✓ ukr_ip (World Bank, interpolated) — {len(ukr_ip.dropna())} obs")
    except Exception as e:
        print(f"  World Bank attempt failed: {e}")

if ukr_ip is not None:
    fred_data["ukr_ip"] = ukr_ip
else:
    print("\n⚠ Could not retrieve Ukraine IP from any source.")
    print("  Manual download required from: https://ukrstat.gov.ua")

## 4. Assemble and save the external dataset

In [ ]:
# Merge all series into a single dataframe
date_range = pd.date_range(START, END, freq="MS")
external = pd.DataFrame(index=date_range)

for name, series in fred_data.items():
    if series is not None:
        external[name] = series.reindex(date_range)

external.index.name = "date"

print("External dataset shape:", external.shape)
print()
print("Coverage by variable:")
for col in external.columns:
    n = external[col].dropna()
    if len(n) > 0:
        print(f"  {col:20s} — {len(n):3d} obs | {n.index.min().date()} to {n.index.max().date()}")
    else:
        print(f"  {col:20s} — NO DATA")

external.to_csv(DATA_EXTERNAL / "external_data.csv")
print("\nSaved: external_data.csv")

## 5. Visualisation — inspect all series before SVAR

In [ ]:
cols_to_plot = [c for c in external.columns if external[c].dropna().shape[0] > 10]
n = len(cols_to_plot)
ncols = 2
nrows = (n + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3))
axes = axes.flatten()

labels = {
    "uah_usd":   "UAH/USD exchange rate",
    "uah_eur":   "UAH/EUR exchange rate",
    "usd_eur":   "USD/EUR",
    "ecb_rate":  "ECB deposit rate (%)",
    "nbu_rate":  "NBU policy rate (%)",
    "brent":     "Brent oil price (USD/bbl)",
    "ea_ip_yoy": "EA industrial production YoY (%)",
    "ukr_ip":    "Ukraine industrial production",
}

for i, col in enumerate(cols_to_plot):
    axes[i].plot(external.index, external[col], lw=1.5, color="steelblue")
    axes[i].set_title(labels.get(col, col), fontsize=10, fontweight="bold")
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axes[i].xaxis.set_major_locator(mdates.YearLocator(4))
    axes[i].tick_params(axis="x", rotation=45)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("External Data — Overview", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(
    Path(cwd).parent.parent / "outputs" / "fig4_external_data.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Figure saved.")

## 6. Data sources documentation

| Variable | Series ID | Source | URL |
|---|---|---|---|
| UAH/USD exchange rate | DEXUAEU | FRED / Federal Reserve | https://fred.stlouisfed.org |
| ECB deposit rate | ECBDFR | FRED / ECB | https://fred.stlouisfed.org |
| Brent crude oil | DCOILBRENTEU | FRED / EIA | https://fred.stlouisfed.org |
| EA industrial production | EA19PRINTO01GYSAM | FRED / Eurostat | https://fred.stlouisfed.org |
| NBU policy rate | — | National Bank of Ukraine | https://bank.gov.ua |
| Ukraine industrial production | — | OECD / IMF IFS / World Bank | https://stats.oecd.org |

All series are downloaded programmatically and require no manual intervention.